# Project 1 - NYC Leading Causes of Death

For this project, I explore NYC's official mortality records using both pandas and pure Python (the "hard way"). Along the way, I calculate the mean, median, and mode of the number of deaths, across categories, and build a small ASCII visualization using only the Python standard library. 

Let's get started :).

## 1. Configuration
Keep all editable setting here to avoid hard-coding later.

In [ ]:
DATA_PATH = "New_York_City_Leading_Causes_of_Death_20251106.csv"
NUMERIC_COL = "Deaths"
FILE_FORMAT = "csv"
MAX_ROWS = None


# For this project, we keep all rows
def row_filter(row):
    return True


print("Config loaded:", DATA_PATH, "| numeric:", NUMERIC_COL)

Config loaded: New_York_City_Leading_Causes_of_Death_20251106.csv | numeric: Deaths


## 2. Dataset overview

**Source:** Bureau of Vital Statistics and New York City Department of Health and Mental Hygiene

_Link to source:_ https://data.cityofnewyork.us/Health/New-York-City-Leading-Causes-of-Death/jb7j-dtam/about_data

Each row in this dataset represents a specific demographic breakdown of mortality:
**a combination of one year, one leading cause of death, one sex group, and one
race/ethnicity group**, with the associated number of deaths recorded in that group.

For example:  
“2021 / Diseases of Heart / Male / Non-Hispanic Black / 1886 deaths”

## 3. Load the data with pandas & quick look

In [4]:
import pandas as pd

df = pd.read_csv(DATA_PATH)
df.head()

,Year,Leading Cause,Sex,Race Ethnicity,Deaths,Death Rate,Age Adjusted Death Rate
0,2021,"Diseases of Heart (I00-I09, I11, I13, I20-I51)",Male,Not Stated/Unknown,190,NaN,NaN
1,2021,Alzheimer's Disease (G30),Female,Not Stated/Unknown,7,NaN,NaN
2,2021,"Diseases of Heart (I00-I09, I11, I13, I20-I51)",Female,Not Stated/Unknown,113,NaN,NaN
3,2021,Malignant Neoplasms (Cancer: C00-C97),Male,Not Stated/Unknown,84,NaN,NaN
4,2021,Cerebrovascular Disease (Stroke: I60-I69),Male,Other Race/ Ethnicity,11,NaN,NaN


### Shape and columns
Confirm the dataset meets the ≥1,000 rows requirement and inspect columns.

In [5]:
print("Shape:", df.shape)
list(df.columns)

Shape: (2102, 7)


['Year',
 'Leading Cause',
 'Sex',
 'Race Ethnicity',
 'Deaths',
 'Death Rate',
 'Age Adjusted Death Rate']

### Cleaning: make the numeric column truly numeric
I coerce `Deaths` to numeric and drop rows where it is missing. The cleaned DataFrame below (`df_clean`) is used for all pandas-based calculations.

In [6]:
df.columns = [c.strip() for c in df.columns]
if NUMERIC_COL not in df.columns:
    raise KeyError(f"{NUMERIC_COL} not in columns: {df.columns[:10]}")

df[NUMERIC_COL] = pd.to_numeric(df[NUMERIC_COL], errors="coerce")
df_clean = df.dropna(subset=[NUMERIC_COL]).copy()

print("Rows after cleaning:", len(df_clean))
df_clean.head()

Rows after cleaning: 1964


,Year,Leading Cause,Sex,Race Ethnicity,Deaths,Death Rate,Age Adjusted Death Rate
0,2021,"Diseases of Heart (I00-I09, I11, I13, I20-I51)",Male,Not Stated/Unknown,190.0,NaN,NaN
1,2021,Alzheimer's Disease (G30),Female,Not Stated/Unknown,7.0,NaN,NaN
2,2021,"Diseases of Heart (I00-I09, I11, I13, I20-I51)",Female,Not Stated/Unknown,113.0,NaN,NaN
3,2021,Malignant Neoplasms (Cancer: C00-C97),Male,Not Stated/Unknown,84.0,NaN,NaN
4,2021,Cerebrovascular Disease (Stroke: I60-I69),Male,Other Race/ Ethnicity,11.0,NaN,NaN


## 4. Mean / Median / Mode

In [7]:
mean_pd = df_clean[NUMERIC_COL].mean()
median_pd = df_clean[NUMERIC_COL].median()
mode_pd = df_clean[NUMERIC_COL].mode()

print("Pandas mean:", mean_pd)
print("Pandas median:", median_pd)
print("Pandas mode(s):", list(mode_pd.values))

Pandas mean: 429.2561099796334
Pandas median: 140.0
Pandas mode(s): [np.float64(1.0)]


**Interpretation.**

The distribution of `Deaths` across cause–year–sex–race groups is highly
right-skewed.

- **Mean ≈ 429 deaths**  
  On average, a single demographic slice (one cause × one year × one sex × one
  race/ethnicity group) records about 429 deaths.

- **Median = 140 deaths**  
  Half of all rows report *fewer than 140* deaths.  
  This big gap between the mean (429) and the median (140) already suggests that
  a small number of categories have extremely high death counts that pull the
  average upward.

- **Mode = 1 death**  
  The most frequently occurring value in the dataset is **1**, meaning that many
  demographic categories experience only a single death for a particular cause
  in a given year.

Taken together, these results indicate a distribution where:
- **most categories report relatively low numbers of deaths**, while  
- **a few categories (e.g., heart disease for certain groups) have extremely
  high counts**.

This is typical of public‐health datasets: common causes of death (heart
disease, cancer) dominate the total burden, while many rare causes generate only
a few deaths in specific demographic groups.

## 5. The hard way 
Re-load the same file using the standard library only and implement mean/median/mode by hand.

In [ ]:
import csv, math, json


def iter_numeric_from_csv(path, column, keep_row=row_filter):
    with open(path, "r", encoding="utf-8", newline="") as f:
        r = csv.DictReader(f)
        for row in r:
            if not keep_row(row):
                continue
            try:
                x = float(row.get(column, ""))
                if math.isfinite(x):
                    yield x
            except (TypeError, ValueError):
                continue


def load_numeric_series(path, column, fmt="csv", limit=None):
    it = iter_numeric_from_json if fmt.lower() == "json" else iter_numeric_from_csv
    vals, i = [], 0
    for v in it(path, column):
        vals.append(v)
        i += 1
        if limit and i >= limit:
            break
    return vals


def mean_py(values):
    return sum(values) / len(values) if values else float("nan")


def median_py(values):
    n = len(values)
    if n == 0:
        return float("nan")
    s = sorted(values)
    mid = n // 2
    return s[mid] if n % 2 == 1 else (s[mid - 1] + s[mid]) / 2


def mode_py(values):
    counts = {}
    for v in values:
        counts[v] = counts.get(v, 0) + 1
    if not counts:
        return []
    maxc = max(counts.values())
    return sorted([v for v, c in counts.items() if c == maxc])


vals = load_numeric_series(DATA_PATH, NUMERIC_COL, FILE_FORMAT, limit=MAX_ROWS)
print("Loaded numeric values:", len(vals))

mean_std = mean_py(vals)
median_std = median_py(vals)
mode_std = mode_py(vals)

print("StdLib mean:", mean_std)
print("StdLib median:", median_std)
print("StdLib mode(s):", mode_std[:10], "(showing up to 10)")

Loaded numeric values: 1964
StdLib mean: 429.2561099796334
StdLib median: 140.0
StdLib mode(s): [1.0] (showing up to 10)


## 6. Visualization
To satisfy the requirement that drawing uses only the standard library, I build a tiny horizontal bar chart. I first aggregate with pandas (allowed for *data/calculation*), then print using pure Python.

In [9]:
year_totals = df_clean.groupby("Year")[NUMERIC_COL].sum().sort_index()
pairs = list(year_totals.items())  # e.g., [('2007', 36287), ...]
pairs[:5]

[(2007, 53996.0),
 (2008, 54138.0),
 (2009, 52820.0),
 (2010, 52505.0),
 (2011, 52726.0)]

In [ ]:
def ascii_bar_chart(pairs, width=40, symbol="▇"):
    vmax = max(v for _, v in pairs) if pairs else 0
    if vmax <= 0:
        print("(no positive values)")
        return
    for label, v in pairs:
        bar = symbol * int(round((v / float(vmax)) * width))
        print(f"{label}: {bar} {v:,}")


print("Deaths by year (ASCII):")
ascii_bar_chart(pairs, width=40)

Deaths by year (ASCII):
2007: ▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇ 53,996.0
2008: ▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇ 54,138.0
2009: ▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇ 52,820.0
2010: ▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇ 52,505.0
2011: ▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇ 52,726.0
2012: ▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇ 52,420.0
2013: ▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇ 53,387.0
2014: ▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇ 53,006.0
2015: ▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇ 54,120.0
2016: ▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇ 54,280.0
2017: ▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇ 54,319.0
2018: ▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇ 55,081.0
2019: ▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇ 54,559.0
2020: ▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇ 82,142.0
2021: ▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇ 63,560.0


## Detours & Dead Ends

At first I planned to analyze `Death Rate` or `Age Adjusted Death Rate`, but
these columns contain many missing values, making them unusable for consistent
calculations.  
Because of this, I switched to focusing on raw `Deaths`, which is complete and
clean enough to support both the pandas and the pure-Python (“hard way”) steps.

## Conclusion, Limitations, and Next Steps

The distribution of `Deaths` across cause–year–sex–race groups is highly
right-skewed. Most demographic slices record relatively small numbers of deaths,
while a few common causes (e.g., heart disease, cancer) dominate the totals.  
This is reflected in the statistics: a **mean (~429)** much higher than the
**median (140)** and a **mode of only 1**.

### Limitations
- These counts are **not population-normalized**; larger demographic groups
  naturally register more deaths.
- Reporting practices and cause definitions may vary by year.
- Missing values in rate columns prevent deeper mortality-risk analysis.

### Next Steps
- Normalize by population to compare groups more meaningfully.
- Analyze trends for specific causes over time.
- Explore whether certain demographic groups experience disproportionate burdens.